# Chapter 3: The Perceptron

> A neuron that only speaks up when it's wrong — and that one rule is enough to learn a hyperplane.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 1 (Decision Trees), Chapter 2 (Geometry & Nearest Neighbors) &nbsp;|&nbsp; **Time:** ~40 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 3

---

## Learning Objectives

- Describe the biological motivation behind the perceptron and its linear activation
- Classify the perceptron as an **online**, **error-driven** learning algorithm
- Implement the perceptron algorithm for binary classification from scratch
- Explain why permuting training examples every epoch matters for convergence speed
- Contrast the vanilla perceptron with the **averaged perceptron** and explain why averaging generalizes better

## The Problem

Decision trees only use a handful of features; KNN uses all features equally. Neither lets you learn *how much* each feature should matter.

The perceptron solves this by learning one weight per feature — turning classification into finding a hyperplane that separates positive from negative examples. Unlike decision trees or KNN, the perceptron is:

- **Online** — it looks at one example at a time
- **Error-driven** — it only updates its weights when it makes a mistake

## The Concept

**The training loop:**

```
Initialize w = 0, b = 0
      │
      ▼
Pick next training example (x, y)
      │
      ▼
Compute activation a = w·x + b
      │
      ▼
   y·a <= 0 ?
   /        \
 yes          no
 (mistake)   (correct)
  │             │
  ▼             ▼
w += y·x      do nothing
b += y          │
  │             │
  └──────┬──────┘
         ▼
 more examples / epochs?
   yes → repeat   no → return w, b
```

### Key Ideas

- **Error-driven:** as long as an example is already correctly classified, the weights don't move — only mistakes trigger an update.
- **The `y·a <= 0` trick:** since labels are ±1, this single check replaces a more verbose "is the sign wrong" comparison.
- **Order matters:** presenting examples in a fixed order (e.g. all positives, then all negatives) can make the perceptron temporarily "forget" how to classify one class. Re-permuting the data every epoch avoids this and tends to converge faster.
- **Linear decision boundary:** the perceptron's boundary is the hyperplane perpendicular to `w`; it can only separate linearly-separable data (it will loop forever on XOR-like problems).
- **Averaging fixes a subtle flaw:** a single late mistake can overwrite weeks of good learning. The averaged perceptron keeps a running average of every weight vector seen during training, so no single late update can dominate the final model.

## Build It

### Setup

NumPy handles the from-scratch math. From scikit-learn we use the Breast Cancer Wisconsin dataset, `StandardScaler` (feature scaling matters here too — perceptron updates are additive in `x`), the reference `Perceptron` classifier to validate against, and `accuracy_score`.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron as SKPerceptron
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(0)

### Step 1: Vanilla Perceptron (Algorithms 5 & 6 in the book)

`fit` runs for `max_iter` epochs. Within each epoch, it walks through the training examples (optionally in a freshly shuffled order — see Section 3.2) and applies the single update rule: if the current weights get an example wrong (`y[n] * a <= 0`), nudge `w` and `b` in the direction of that example's label.

`predict` just returns the sign of the linear activation `w·x + b`.

In [2]:
class PerceptronFromScratch:
    def __init__(self, max_iter=50, permute=True, random_state=0):
        self.max_iter = max_iter
        self.permute = permute
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        N, D = X.shape
        self.w = np.zeros(D)
        self.b = 0.0
        rng = np.random.RandomState(self.random_state)

        order = np.arange(N)
        for _ in range(self.max_iter):
            if self.permute:
                rng.shuffle(order)
            for n in order:
                a = self.w @ X[n] + self.b
                if y[n] * a <= 0:
                    self.w += y[n] * X[n]
                    self.b += y[n]
        return self

    def decision_function(self, X):
        return np.asarray(X) @ self.w + self.b

    def predict(self, X):
        return np.sign(self.decision_function(X))

### Step 2: Averaged Perceptron (Algorithm 7 in the book)

The averaged perceptron runs the exact same mistake-driven updates, but also maintains a running cache (`u`, `beta`) that lets it compute the **average** of every weight vector seen across all of training — without ever materializing that full history. The final `self.w` and `self.b` are recovered via a telescoping-sum trick at the end of `fit`, rather than by literally averaging a huge list of vectors.

In [3]:
class AveragedPerceptronFromScratch:
    def __init__(self, max_iter=50, permute=True, random_state=0):
        self.max_iter = max_iter
        self.permute = permute
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        N, D = X.shape
        w = np.zeros(D)
        b = 0.0
        u = np.zeros(D)
        beta = 0.0
        c = 1
        rng = np.random.RandomState(self.random_state)

        order = np.arange(N)
        for _ in range(self.max_iter):
            if self.permute:
                rng.shuffle(order)
            for n in order:
                if y[n] * (w @ X[n] + b) <= 0:
                    w += y[n] * X[n]
                    b += y[n]
                    u += y[n] * c * X[n]
                    beta += y[n] * c
                c += 1

        self.w = w - u / c
        self.b = b - beta / c
        return self

    def decision_function(self, X):
        return np.asarray(X) @ self.w + self.b

    def predict(self, X):
        return np.sign(self.decision_function(X))


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

## Use It — Real Data

### Experiment A: Sanity Check vs. `sklearn.linear_model.Perceptron`

We load the Breast Cancer Wisconsin dataset, remap labels to the book's ±1 convention, split into train/test, and scale features. Then we compare our from-scratch perceptron against scikit-learn's reference implementation.

An exact match is **not** expected here: both algorithms are order-sensitive and rely on non-convex-style mistake-driven optimization, so different internal tie-breaking and update schedules can lead to slightly different final weights.

In [4]:
data = load_breast_cancer()
X, y_raw = data.data, data.target
y = np.where(y_raw == 0, -1, 1)
print(f"Dataset shape: {X.shape[0]} examples, {X.shape[1]} features")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

my_perc = PerceptronFromScratch(max_iter=20, permute=True, random_state=1).fit(X_train_s, y_train)
my_pred = my_perc.predict(X_test_s)
my_acc = accuracy_score(y_test, my_pred)

sk_perc = SKPerceptron(max_iter=20, tol=None, random_state=1).fit(X_train_s, y_train)
sk_pred = sk_perc.predict(X_test_s)
sk_acc = accuracy_score(y_test, sk_pred)

print(f"From-scratch Perceptron test accuracy : {my_acc:.4f}")
print(f"sklearn Perceptron test accuracy      : {sk_acc:.4f}")

Dataset shape: 569 examples, 30 features
From-scratch Perceptron test accuracy : 0.9591
sklearn Perceptron test accuracy      : 0.9532


### Experiment B: Permuting Data Each Epoch vs. Fixed Order (Section 3.2)

Here we test the book's claim directly: does re-shuffling the training order every epoch actually help the perceptron learn faster (reach higher training accuracy in fewer epochs), compared to always visiting examples in the same fixed order?

In [5]:
print(f"{'epochs':>7} | {'no permute (train acc)':>24} | {'permute each epoch (train acc)':>32}")
print("-" * 68)
for epochs in [1, 2, 5, 10, 20]:
    p_fixed = PerceptronFromScratch(max_iter=epochs, permute=False, random_state=1).fit(X_train_s, y_train)
    p_perm = PerceptronFromScratch(max_iter=epochs, permute=True, random_state=1).fit(X_train_s, y_train)
    acc_fixed = accuracy(y_train, p_fixed.predict(X_train_s))
    acc_perm = accuracy(y_train, p_perm.predict(X_train_s))
    print(f"{epochs:>7} | {acc_fixed:>24.4f} | {acc_perm:>32.4f}")

 epochs |   no permute (train acc) |   permute each epoch (train acc)
--------------------------------------------------------------------
      1 |                   0.9698 |                           0.9673
      2 |                   0.9824 |                           0.9673
      5 |                   0.9698 |                           0.9824
     10 |                   0.9799 |                           0.9698
     20 |                   0.9824 |                           0.9849


### Experiment C: Overfitting vs. `MaxIter` (Section 3.2, Figure 3.3)

Just like `max_depth` for decision trees and `K` for KNN, `MaxIter` (the number of epochs) is a hyperparameter that trades off underfitting and overfitting. We expect training accuracy to climb steadily while test accuracy eventually drops — the classic signature that motivates **early stopping**.

In [6]:
print(f"{'MaxIter':>8} | {'train acc':>10} | {'test acc':>9}")
print("-" * 34)
for it in [1, 2, 5, 10, 20, 50, 100, 200]:
    p = PerceptronFromScratch(max_iter=it, permute=True, random_state=1).fit(X_train_s, y_train)
    tr_acc = accuracy(y_train, p.predict(X_train_s))
    te_acc = accuracy(y_test, p.predict(X_test_s))
    print(f"{it:>8} | {tr_acc:>10.4f} | {te_acc:>9.4f}")

 MaxIter |  train acc |  test acc
----------------------------------
       1 |     0.9673 |    0.9708
       2 |     0.9673 |    0.9591
       5 |     0.9824 |    0.9766
      10 |     0.9698 |    0.9591
      20 |     0.9849 |    0.9591
      50 |     0.9849 |    0.9532
     100 |     0.9925 |    0.9708


     200 |     0.9874 |    0.9415


**Reading the table:** as `MaxIter` grows, training accuracy tends toward 1.0, while test accuracy can start to fall off after some point — reproducing the overfitting curve described in the book's Figure 3.3.

### Experiment D: Vanilla vs. Averaged Perceptron (Section 3.6)

Finally, we test the book's central claim about averaging: at a given `MaxIter`, does the averaged perceptron degrade less than the vanilla perceptron once training has gone on long enough to start overfitting?

In [7]:
print(f"{'MaxIter':>8} | {'vanilla test acc':>17} | {'averaged test acc':>18}")
print("-" * 48)
for it in [1, 5, 20, 50, 100, 200]:
    p_van = PerceptronFromScratch(max_iter=it, permute=True, random_state=1).fit(X_train_s, y_train)
    p_avg = AveragedPerceptronFromScratch(max_iter=it, permute=True, random_state=1).fit(X_train_s, y_train)
    acc_van = accuracy(y_test, p_van.predict(X_test_s))
    acc_avg = accuracy(y_test, p_avg.predict(X_test_s))
    print(f"{it:>8} | {acc_van:>17.4f} | {acc_avg:>18.4f}")

 MaxIter |  vanilla test acc |  averaged test acc
------------------------------------------------
       1 |            0.9708 |             0.9532
       5 |            0.9766 |             0.9708
      20 |            0.9591 |             0.9708


      50 |            0.9532 |             0.9708
     100 |            0.9708 |             0.9766


     200 |            0.9415 |             0.9649


**Reading the table:** as `MaxIter` grows large, the vanilla perceptron's test accuracy tends to degrade noticeably more than the averaged perceptron's — confirming Section 3.6's claim that averaging is measurably more robust to overtraining.

## Use It

| API / Function | When to use it |
|---|---|
| `PerceptronFromScratch(max_iter, permute).fit(X, y).predict(Xtest)` | Simple, fast, online linear classifier on separable-ish data |
| `AveragedPerceptronFromScratch(max_iter, permute).fit(X, y)` | Same as above but more robust to overtraining — prefer this in practice |
| `sklearn.linear_model.Perceptron` | Production use — has L1/L2 regularization options and optimized solvers |
| `StandardScaler` (sklearn) | Apply before training — perceptron updates are additive in `x`, so unscaled features distort the geometry |

## Exercises

1. Modify the training loop to record the number of mistakes made per epoch, and plot it — do mistakes decrease monotonically?
2. Implement the **voted perceptron** (Eq. 3.17 in the book): instead of averaging weight vectors, store every intermediate `(w, b)` along with its survival count, and have each one "vote" at test time. Compare its test accuracy and prediction time to the averaged perceptron.
3. Run the perceptron on a synthetic XOR-style dataset (4 points, not linearly separable) and confirm it never converges — cap the epochs and observe the weights oscillating instead of stabilizing.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Perceptron** | "Just a simple neuron toy" | An online, error-driven linear classifier whose decision boundary is the hyperplane perpendicular to its learned weight vector |
| **Margin** | "Just the gap between classes" | The distance from the separating hyperplane to the closest training point; the perceptron's convergence speed is bounded by `1/margin²` |
| **Linearly Separable** | "Any two-class dataset" | A dataset for which *some* hyperplane perfectly separates the classes; the perceptron is only guaranteed to converge if this holds |
| **Averaging** | "A minor implementation detail" | A regularization-like technique that returns the mean of all weight vectors seen during training, reducing sensitivity to the order and recency of examples |

## Summary

- The perceptron learns one weight per feature via an **online, error-driven** update rule
- It only updates on mistakes: `y·a <= 0` triggers `w += y·x`, `b += y`
- **Re-permuting the data each epoch** tends to speed up convergence compared to a fixed order
- `MaxIter` (epochs) trades off underfitting and overfitting, just like `max_depth` and `K` did in earlier chapters
- The perceptron can only separate **linearly separable** data — it loops forever otherwise
- The **averaged perceptron** returns the mean of all weight vectors seen during training and generalizes measurably better than the vanilla version, especially with more epochs

---

**Next:** Chapter 4 — Practical Issues in Machine Learning